# Self-correction(Proposed Method)

In [2]:
import numpy as np
import pandas as pd
import random
import ollama
from tqdm import tqdm
import re
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [3]:
ollama_model = "llama3:70b-instruct"

In [ ]:
# Generated instructions(Load Instruction Induction Prompt)
instruction_cleaned = pd.read_csv('inductions.csv')
instruction_list = list(instruction_cleaned['induction_set'])

# 1st Eval
test = pd.read_csv('evaluation_1.csv')

test_q = list(test["question"])
test_a = list(test["essay"])
test_output = list(test["domain1_score"])

# Predicted Score(LLM Output)
file_path = 'firstresult.csv'
df = pd.read_csv(file_path)
prediction = {idx: row for idx, row in df.iterrows()}

# Actual Score
answer = list(test["domain1_score"])

# Error Extraction and Prompt Construction

In [9]:
def get_false(predic_list, false_index):
    return [predic_list[i] for i in false_index]

In [ ]:
predics_final = {i: list(pred) for i, pred in prediction.items()}

In [11]:
all_input_false_value_q = []
all_input_false_value_a = []
all_answer_false_value = []
all_predic_false_value = []

In [12]:
for key, predictions in predics_final.items():
    diff = [pred == real for pred, real in zip(predictions, answer)]
    false_indices = [i for i, is_correct in enumerate(diff) if not is_correct]
    
    all_input_false_value_q.append(get_false(test_q, false_indices))
    all_input_false_value_a.append(get_false(test_a, false_indices))
    all_answer_false_value.append(get_false(answer, false_indices))
    all_predic_false_value.append(get_false(predictions, false_indices))

In [13]:
wrong_shots = [
    "\n\n".join(
        f"Question: {input_q}\nEssay: {input_a}\nWrong score: {predic_val}\nActual score: {answer_val}"
        for input_q, input_a, predic_val, answer_val in zip(input_false_q, input_false_a, predic_false, answer_false)
    )
    for input_false_q, input_false_a, predic_false, answer_false in zip(all_input_false_value_q, all_input_false_value_a, all_predic_false_value, all_answer_false_value)
]

In [ ]:
# Prompt Format(IELTS Writing)
all_prompt = []
for i in range(len(instruction_list)):
    prompt = f"""The IELTS Writing Task 2 examiner scored the answer essays using the following instructions.
 Base Instruction: {instruction_list[i]}

 Some of the answer essays were scored correctly, but others were not assessed against the criteria for IELTS Writing Task 2 (task response, coherence and cohesion, vocabulary material, grammatical range and accuracy).
 
 The examiner should score an answer essay based on the following Criteria.
   *IELTS Writing Task 2's specific Key Assessment Criteria*
        Score 9
        - Task response: The answer is fully and deeply addressed, with a clear and well-developed position directly answering the question. Ideas are relevant, fully extended, and well-supported.
        - Coherence and cohesion: The message flows effortlessly, with minimal lapses in coherence and cohesion. Paragraphing is managed skillfully.
        - Lexical resource: Vocabulary is used flexibly and precisely, with very few minor errors.
        - Grammatical range and accuracy: A wide range of structures is used with full control. Minor errors are extremely rare.

        Score 8
        - Task response: The answer is appropriately and sufficiently addressed, with a clear, well-developed position. Ideas are relevant, well-extended, and supported, though occasional lapses may occur.
        - Coherence and cohesion: The message is easy to follow, with logical sequencing and well-managed cohesion. Occasional minor lapses may occur.
        - Lexical resource: Vocabulary is used fluently and flexibly, with minor errors that don't impact communication.
        - Grammatical range and accuracy: A wide range of structures is used accurately, with occasional minor errors.

        Score 7
        - Task response: The main parts of the answer are addressed, with a clear and developed position. Ideas are extended and supported but may lack focus or precision.
        - Coherence and cohesion: Information is logically organized, with clear progression, though minor lapses may occur. Cohesion devices are used flexibly but with some inaccuracies.
        - Lexical resource: Vocabulary is sufficient for flexibility and precision, with occasional errors.
        - Grammatical range and accuracy: A variety of complex structures is used with general accuracy, though some errors may persist.

        Score 6
        - Task response: The main parts of the answer are addressed, but some may be more fully covered than others. Ideas may lack clarity or be underdeveloped.
        - Coherence and cohesion: Information is generally coherent with clear progression, but cohesion may be faulty or mechanical at times.
        - Lexical resource: Vocabulary is generally adequate, though range may be limited, and errors may occur.
        - Grammatical range and accuracy: A mix of simple and complex sentences is used, but errors in grammar and punctuation are more frequent.

        Score 5
        - Task response: The answer is incompletely addressed, with limited development of ideas. There may be repetition and irrelevant details.
        - Coherence and cohesion: Organization is evident but not wholly logical, with issues in sentence fluency and paragraphing.
        - Lexical resource: Vocabulary is limited, with frequent errors and lack of variation.
        - Grammatical range and accuracy: The range of structures is limited, with frequent grammatical errors that may affect understanding.

 Below are the Question and Answer Essay pairs that were incorrectly scored with the corresponding base instruction, the incorrect scoring score, and the correct scoring score based on the actual assessment criteria.
 
 {wrong_shots[i]}

 Modify the Instruction to let examiners correctly score the Answer Essay to accurately reflect the IELTS Writing Task 2 assessment criteria."""
    
    all_prompt.append(prompt)

In [ ]:
# ASAP++ Assessment Criteria
    # Score 3
    #         - Content: The response answers the question asked of it. Sufficient evidence from the story isused to support the points that the writer makes.
    #         - Question Adherence: The response shows an excellent understanding of the meaning of the text andquestion, and stays on topic.
    #         - Language: Grammar and spelling are excellent, with a wide range of grammatical structures used.The writing shows evidence of a high range of vocabulary, with words used to good effect in appropriate places.
    #         - Narrativity: The response is interesting. Appropriate use of transitional and linking words and sentences makes the narrative flow smoothly. It is often conversational and makes the story easy to follow.

    #         Score 2
    #         - Content: The response addresses some of the points. Evidence from the story supporting those points are present.
    #         - Question Adherence: The response shows a good understanding of the meaning of the text and question,and occasionally wanders off topic.
    #         - Language: Grammar and spelling are good, with only some minor errors. Different kinds of grammatical structures may be used. The writing shows evidence of an adequate range of vocabulary.
    #         - Narrativity: The response is somewhat interesting. Transitional and linking words are used in some places, but not everywhere.

    #         Score 1
    #         - Content: The response may lack information / evidence showing a lack of understanding of the text.
    #         - Question Adherence: The response shows a misreading of the text or question, or consistently wanders off topic.
    #         - Language: Grammar and spelling show many errors. Vocabulary is limited and not very varied.Some words may be used in inappropriate places.
    #         - Narrativity: The response is very uninteresting and disjointed and is unable to deliver the content at all.

    #         Score 0
    #         - Content: The response is irrelevant / incorrect / incomplete.
    #         - Question Adherence: The response is irrelevant / incorrect / incomplete.
    #         - Language: There are spelling and grammar errors in almost every sentence. Vocabulary isextremely limited, leading to repetitive use of words, as well as incorrect use of words, in many places.
    #         - Narrativity: The response is irrelevant / incorrect / incomplete.

In [ ]:
# ELLIPSE Assessment Criteria
    # Score 5:
    # - Cohesion: Text organization consistently well controlled using a variety of effective linguistic features such as reference and transitional words and phrases to connect ideas across sentences and paragraphs; appropriate overlap of ideas.
    # - Syntax: Flexible and effective use of a full range of syntactic structures including simple, compound, and complex sentences; There may be rare minor and negligible errors in sentence formation.
    # - Vocabulary: Wide range of vocabulary flexibly and effectively used to convey precise meanings; skillful use of topic-related terms and less common words; rare negligible inaccuracies in word use.
    # - Phraseology: Flexible and effective use of a variety of phrases, such as idioms, collocations, and lexical bundles, to convey precise and subtle meanings; rare minor inaccuracies that are negligible.
    # - Grammar: Command of grammar and usage with few or no errors.
    # - Conventions: Consistent use of appropriate conventions to convey meaning; spelling, capitalization, and punctuation errors nonexistent or negligible.

    # Score 4:
    # - Cohesion: Organization generally well controlled; a range of cohesive devices used appropriately such as reference and transitional words and phrases to connect ideas; generally appropriate overlap of ideas.
    # - Syntax: Appropriate use of a variety of syntactic structures, such as simple, compound, and complex sentences; occasional errors or inappropriateness in sentence formation.
    # - Vocabulary: Sufficient range of vocabulary to allow flexibility and precision; appropriate use of topic-related terms and less common lexical items.
    # - Phraseology: Appropriate use of a variety of phrases, such as idioms, collocations, and lexical bundles; occasional inaccuracies and colloquialisms.
    # - Grammar: Minimal errors in grammar and usage.
    # - Conventions: Generally consistent use of appropriate conventions to convey meaning; spelling, capitalization, and punctuation errors few and not distracting.

    # Score 3:
    # - Cohesion: Organization generally controlled; cohesive devices used but limited in type; Some repetitive, mechanical, or faulty use of cohesion use within and/or between sentences and paragraphs.
    # - Syntax: Simple, compound, and complex syntactic structures present although the range may be limited; some apparent errors in sentence formation, especially in more complex sentences.
    # - Vocabulary: Minimally adequate range of vocabulary for the topic; no precise use of subtle word meanings; topic related terms only used occasionally; attempts to use less common vocabulary but with some inaccuracy.
    # - Phraseology: Evident use of phrases such as idioms, collocations, and lexical bundles but without much variety; some noticeable repetitions and misuses.
    # - Grammar: Some errors in grammar and usage.
    # - Conventions: Developing use of conventions to convey meaning; errors in spelling, capitalization, and punctuation that are sometimes distracting.

    # Score 2:
    # - Cohesion: Organization only partially developed with a lack of logical sequencing of ideas; some basic cohesive devices used but with inaccuracy or repetition.
    # - Syntax: Some sentence variation used; many sentence structure problems.
    # - Vocabulary: Narrow range of vocabulary to convey basic and elementary meaning; topic related terms used inappropriately; errors in word formation and word choice that may distort meanings.
    # - Phraseology: Narrow range of phrases, such as collocations and lexical bundles, used to convey basic and elementary meaning; many repetitions and/or misuses of phrases.
    # - Grammar: Many errors in grammar and usage.
    # - Conventions: Variable use of conventions; spelling, capitalization, and punctuation errors frequent and distracting.

    # Score 1:
    # - Cohesion: No clear control of organization; cohesive devices not present or unsuccessfully used; presentation of ideas unclear.
    # - Syntax: Pervasive and basic errors in sentence structure and word order that cause confusion; basic sentences errors common.
    # - Vocabulary: Limited vocabulary often inappropriately used; limited control of word choice and word forms; little attempt to use topic-related terms.
    # - Phraseology: Memorized chunks of language, or simple phrasal patterns predominate; many repetitions and misuses of phrases.
    # - Grammar: Errors in grammar and usage throughout.
    # - Conventions: Minimal use of conventions; spelling, capitalization, and punctuation errors throughout.

# Start Self-Correction

In [ ]:
for4inst = []

for prompt in all_prompt:
    
    response = ollama.chat(
        model=ollama_model,
        messages=[
        {"role": "system", "content": "You are a scoring rubric expert. Improve the instruction to align with the official Essay Scoring Guidelines. Be specific, objective, and actionable. Output only the revised instruction."},
        {'role': 'user', 'content': prompt}],
        )
    print(response['message']['content'])
    for4inst.append(response['message']['content'])

In [19]:
modified_instructions_df = pd.DataFrame(for4inst)
modified_instructions_df.to_csv('selfcorr_modified.csv')

# Second Evaluation

In [ ]:
# Load Self-Corrected Instructions
self_correction = pd.read_csv('selfcorr_modified.csv')
self_correction_list = list(self_correction['revise'])

In [ ]:
# 2nd eval data
test = pd.read_csv('evaluation_2.csv')
test_qq = list(test["question"])
test_aa = list(test["essay"])
test_output = list(test["domain1_score"])

In [ ]:
# Prompt Format
self_correction_instruction_dic = {}
number = 0

for instruction in self_correction_list:
    input_prompt = []
    
    for question, answer in zip(test_qq, test_aa):
        prompt = f"""{instruction}

Question: {question}
Answer Essay: {answer}
Score: """
        input_prompt.append(prompt)
    
    self_correction_instruction_dic[number] = input_prompt
    
    number = number + 1

In [ ]:
# Start Evaluation
result10 = {}

for i in tqdm(range(len(self_correction_instruction_dic))):
    
    prompt_list = self_correction_instruction_dic[i]
    result_list = []
    
    for each_prompt in prompt_list:
        response = ollama.chat(
            model=ollama_model,
            messages=[
                {"role": "system", "content": "You are a helpful chatbot of this scoring task. Generate only an overall band score without further explanation. Generate the essay score as an integer."},
                {'role': 'user', 'content': each_prompt}],
            )
        print(response['message']['content'])
        result_list.append(response['message']['content'])
    
    result10[i] = result_list

In [ ]:
result_df10 = pd.DataFrame.from_dict(result10, orient='index')
result_df10.to_excel('self_corr_result.xlsx')

# Scoring

In [ ]:
test_cc = pd.read_excel('evaluation_2.xlsx')
test_ee = pd.read_excel('self_corr_result.xlsx', engine='openpyxl')

In [ ]:
test_cc_list = test_cc['domain1_score'].tolist()
test_ee_lists = test_ee.values.tolist()

In [ ]:
# QWK
qwk_list = []

from sklearn.metrics import cohen_kappa_score

for clean_test_e_list in test_ee_lists:
    kappa_quadratic = cohen_kappa_score(test_cc_list, clean_test_e_list, weights='quadratic', labels=[5,6,7,8,9])
    qwk_list.append(kappa_quadratic)
# IELTS labels=[5,6,7,8,9], ASAP++ labels=[0,1,2,3], ELLIPSE labels=[1,2,3,4,5]

In [ ]:
# Executive Accuracy
score_dict4 = {}
score_list4 = []

for i, clean_test_e_list in enumerate(test_ee_lists):
    diff = [pred == real for pred, real in zip(clean_test_e_list, test_cc_list)]
    score_list4.append(sum(diff))
    score_dict4[i] = diff

In [36]:
# mse
mse_scores = []

for clean_test_e_list in test_ee_lists:
    mse = mean_squared_error(test_cc_list, clean_test_e_list)
    mse_scores.append(mse)

In [37]:
# rmse
#rmse
rmse_scores = []

for clean_test_e_list in test_ee_lists:
    mse = mean_squared_error(test_cc_list, clean_test_e_list)
    rmse = np.sqrt(mse)
    rmse_scores.append(rmse)